# Abdomen CT — nnUNet v2 — **Faz 2b: Eğitim + Değerlendirme**

**Zorunlu dataset (Faz2a output'u):**
- `nnunet-preprocessed` → `nnunet_preprocessed/`, `splits_out/`, `nifti_val/` içerir

**Opsiyonel dataset'ler:**
- `nnunet-checkpoint` → önceki session checkpoint (devam için)
- `abdomen` → ham DICOM (yalnızca `nifti_val/` eksikse)

---

| Adım | Süre (T. yakl.) |
|------|------------------|
| nnUNet eğitim 3d_fullres | 8–12 saat |
| Inference + Değerlendirme | 20–30 dk |

In [1]:
import json
from pathlib import Path
import os

kaggle_json_content = """
{"username":"ramazan2020","key":"f0aa1fb5d28a061fd73a8552ed0eb043"}
"""

# Check if the user has replaced the placeholder content
if kaggle_json_content.strip() == "" or "PASTE YOUR KAGGLE.JSON CONTENT HERE" in kaggle_json_content:
    raise ValueError("Please paste your kaggle.json content into the 'kaggle_json_content' variable.")

KAGGLE_DIR = Path.home() / '.kaggle'
KAGGLE_JSON_PATH = KAGGLE_DIR / 'kaggle.json'

# Create the .kaggle directory if it doesn't exist
KAGGLE_DIR.mkdir(parents=True, exist_ok=True)

try:
    # Ensure the content is valid JSON
    json.loads(kaggle_json_content)
    # Write the content to the kaggle.json file
    KAGGLE_JSON_PATH.write_text(kaggle_json_content.strip())
    # Set appropriate permissions for the file
    KAGGLE_JSON_PATH.chmod(0o600)
    print(f"'kaggle.json' created at {KAGGLE_JSON_PATH} with provided content.")
    print("You can now run the next cell (q4n0hJXmecfb) to set up Kaggle API credentials.")
except json.JSONDecodeError:
    raise ValueError("Invalid JSON content provided for kaggle.json. Please ensure it's a valid JSON string.")
except Exception as e:
    print(f"Error creating kaggle.json: {e}")

'kaggle.json' created at /root/.kaggle/kaggle.json with provided content.
You can now run the next cell (q4n0hJXmecfb) to set up Kaggle API credentials.


In [2]:
import os
import json
from pathlib import Path

# Define the path for the Kaggle configuration
KAGGLE_DIR = Path.home() / '.kaggle'
KAGGLE_JSON_PATH = KAGGLE_DIR / 'kaggle.json'

# Create the .kaggle directory if it doesn't exist
KAGGLE_DIR.mkdir(parents=True, exist_ok=True)

# Check if kaggle.json is already in the correct location
if not KAGGLE_JSON_PATH.exists():
    # Assume kaggle.json was uploaded to the current content directory
    uploaded_kaggle_json = Path('./kaggle.json')
    if uploaded_kaggle_json.exists():
        uploaded_kaggle_json.rename(KAGGLE_JSON_PATH)
        print(f"Moved 'kaggle.json' to {KAGGLE_JSON_PATH}")
    else:
        raise FileNotFoundError(
            "kaggle.json not found. Please upload your kaggle.json file "
            "to the Colab session storage and re-run this cell."
        )
else:
    print(f"'kaggle.json' already exists at {KAGGLE_JSON_PATH}")

# Set permissions for kaggle.json
KAGGLE_JSON_PATH.chmod(0o600)

# Load credentials and set environment variables
try:
    with open(KAGGLE_JSON_PATH, 'r') as f:
        kaggle_creds = json.load(f)
    os.environ['KAGGLE_USERNAME'] = kaggle_creds['username']
    os.environ['KAGGLE_KEY'] = kaggle_creds['key']
    print("Kaggle API credentials loaded successfully.")
except Exception as e:
    print(f"Error loading Kaggle credentials: {e}")
    print("Please ensure your kaggle.json is correctly formatted and located.")

# Install Kaggle API client (if not already installed)
!pip install -q kaggle

print("\n--- Kaggle Dataset Download Example ---")
print("You can now download datasets from Kaggle. Here's an example:")
print("Replace 'dataset-owner/dataset-name' with the actual dataset you want.")
print("For example, 'kaggle datasets download -d ramazan2020/abdomen'\n")

# Example: Download a public dataset (uncomment and modify as needed)
# !kaggle datasets download -d tensorflow/tf-flowers -p /content/datasets/tf-flowers --unzip

# Another example related to the notebook's context (if applicable)
# To download the 'abdomen' dataset from ramazan2020:
# !kaggle datasets download -d ramazan2020/abdomen -p /content/kaggle_data --unzip

print("Kaggle setup complete. You can now use the `!kaggle` command to interact with Kaggle.")

'kaggle.json' already exists at /root/.kaggle/kaggle.json
Kaggle API credentials loaded successfully.

--- Kaggle Dataset Download Example ---
You can now download datasets from Kaggle. Here's an example:
Replace 'dataset-owner/dataset-name' with the actual dataset you want.
For example, 'kaggle datasets download -d ramazan2020/abdomen'

Kaggle setup complete. You can now use the `!kaggle` command to interact with Kaggle.


In [ ]:
!kaggle datasets download -d ramazan2020/abdomen-nnunet -p /content/kaggle_data --unzip

Dataset URL: https://www.kaggle.com/datasets/ramazan2020/abdomen-nnunet
License(s): unknown
 63% 13.0G/20.4G [01:42<00:56, 141MB/s]

---
## 0. Ortam ve GPU

In [ ]:
import os, sys, subprocess
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB  = not IS_KAGGLE and os.path.exists('/content')
IS_LOCAL  = not IS_KAGGLE and not IS_COLAB
env_name  = 'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Local'
print(f'Ortam : {env_name}')

import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU bulunamadı! Kaggle: Settings→Accelerator→GPU')
print(f'GPU   : {torch.cuda.get_device_name(0)}')
print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'CUDA  : {torch.version.cuda}')

---
## 1. Kütüphane Kurulumu

In [ ]:
import importlib, shutil, sysconfig

# ─── P100 (sm_60) + PyTorch uyumluluk notu ──────────────────────────────
# PyTorch >= 2.6 artık yalnızca sm_70+ destekler; P100 (sm_60) dışlandı.
# sm_60 destekli son sürümler torch 2.4.x ve altıdır.
#
# Çözüm A (kod): sm_60 içeren eski torch kur (aşağıdaki döngü)
# Çözüm B (önerilir): Kaggle → Settings → Accelerator → GPU T4 x2 seç
#   T4 (sm_75) modern torch ile tamamen uyumludur, 16 GB VRAM, daha hızlı.

# 1. Uyumlu torch'u ÖNCE kur (nnunetv2'nin üzerine yazmasını önlemek için)
print("P100 (sm_60) uyumlu torch aranıyor...")
_torch_ok     = False
_torch_pkg    = None
_torch_idx    = None
for _cuda, _pkgs in [
    ('cu118', ['torch==2.4.1+cu118', 'torch==2.3.1+cu118', 'torch==2.2.2+cu118']),
    ('cu121', ['torch==2.4.1+cu121', 'torch==2.3.1+cu121', 'torch==2.2.2+cu121']),
]:
    if _torch_ok:
        break
    _idx = f'https://download.pytorch.org/whl/{_cuda}'
    for _pkg in _pkgs:
        print(f"  {_pkg} ...", end=' ', flush=True)
        _r = subprocess.run(
            [sys.executable, '-m', 'pip', 'install',
             '--force-reinstall', '--no-deps',
             '--index-url', _idx, _pkg],
            capture_output=True, text=True
        )
        if _r.returncode == 0:
            print("✓")
            _torch_ok  = True
            _torch_pkg = _pkg
            _torch_idx = _idx
            break
        else:
            _msg = (_r.stdout + _r.stderr).strip()
            print("✗")
            if _msg:
                print(f"    → {_msg[-200:]}")

if not _torch_ok:
    print()
    print("⚠ P100 (sm_60) uyumlu torch KURULAMADI.")
    print("  En kolay çözüm: GPU tipini T4 ile değiştirin.")
    print("  Notebook Settings → Accelerator → GPU T4 x2")

# 2. nnunetv2 + bağımlılıklar (torch hariç yükler idealdir, ama pin garantisi yok)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'nnunetv2', 'SimpleITK', 'pydicom', 'scipy', 'tqdm'],
    check=True
)
print("nnunetv2 kuruldu.")

# 3. nnunetv2 torch'u yükseltmiş olabilir → uyumlu sürümü geri yükle
if _torch_pkg and _torch_idx:
    _r2 = subprocess.run(
        [sys.executable, '-m', 'pip', 'install',
         '--force-reinstall', '--no-deps',
         '--index-url', _torch_idx, _torch_pkg],
        capture_output=True, text=True
    )
    if _r2.returncode == 0:
        print(f"torch {_torch_pkg.split('==')[1]} yeniden sabitlendl.")
    else:
        print(f"torch yeniden yükleme: {(_r2.stdout + _r2.stderr)[-200:]}")

importlib.invalidate_caches()

# 4. Subprocess GPU testi — yeni kurulan torch ile (kernel cache'ini bypass eder)
_gpu_test = subprocess.run(
    [sys.executable, '-c',
     'import torch; '
     'print(f"torch {torch.__version__}  CUDA {torch.version.cuda}"); '
     't = torch.zeros(1, device="cuda") + 1; '
     'cap = torch.cuda.get_device_capability(); '
     'print(f"GPU: {torch.cuda.get_device_name(0)}  sm_{cap[0]}{cap[1]}  → OK")'],
    capture_output=True, text=True
)
if _gpu_test.returncode == 0:
    print(_gpu_test.stdout.strip())
else:
    print("GPU testi BAŞARISIZ:")
    print((_gpu_test.stdout + _gpu_test.stderr)[-600:])
    if not _torch_ok:
        print("\n→ GPU'yu T4 ile değiştirip notebook'u yeniden başlatın.")
        print("  Notebook Settings → Accelerator → GPU T4 x2")

def find_nnunet_cmd(name: str) -> str:
    p = shutil.which(name)
    if p: return p
    for d in [sysconfig.get_path('scripts'), str(Path(sys.executable).parent),
              '/usr/local/bin', '/opt/conda/bin']:
        c = Path(d) / name
        if c.exists(): return str(c)
    return name

for cmd in ['nnUNetv2_train', 'nnUNetv2_predict']:
    print(f'  {cmd}: {Path(find_nnunet_cmd(cmd)).exists()}')

In [ ]:
!pip install --upgrade git+https://github.com/FabianIsensee/hiddenlayer.git


---
## 2. Konfigürasyon

In [ ]:
import os, json, shutil, time
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List

# ─── Eğitim ayarları ───────────────────────────────────────────────────────
CONTINUE_TRAINING  = False  # True → mevcut checkpoint'ten devam
FOLD               = 0
DATASET_ID         = 100
DATASET_NAME       = 'Abdomen'
GITHUB_URL         = 'https://github.com/ramazan2020/abdomen1.git'

CHECKPOINT_URL = ''   # CONTINUE_TRAINING=True ise önceki session zip URL'si

SUPER_CLASSES: List[str] = [
    'acute_cholecystitis', 'kidney_ureter_stone', 'acute_pancreatitis',
    'aortic_aneurysm_dissection', 'acute_appendicitis', 'acute_diverticulitis',
]

# ─── Kaggle dataset slug'ları ───────────────────────────────────────────────
KAGGLE_DATASET_SLUG = 'abdomen'          # ham DICOM + Bilgi.xlsx
PRE_DATASET_SLUG    = 'abdomen-nnunet'   # Faz2a output — Kaggle'daki slug ile eşleşmeli
# ────────────────────────────────────────────────────────────────────────────

if IS_KAGGLE:
    WORK_DIR  = Path('/kaggle/working')
    TMP_DIR   = Path('/kaggle/tmp')
    _base     = Path('/kaggle/input/datasets/ramazan2020')
    DATA_DIR  = _base / KAGGLE_DATASET_SLUG
    _pre_base = _base / PRE_DATASET_SLUG

    if not _pre_base.exists():
        raise FileNotFoundError(
            f'Zorunlu dataset bulunamadı: {_pre_base}\n'
            f'Sağ panelden "{PRE_DATASET_SLUG}" (Faz2a output) ekleyin.\n'
            f'Dataset slug farklıysa PRE_DATASET_SLUG değişkenini güncelleyin.'
        )

    # Input'taki preprocessed yolu (read-only)
    _pre_input = next(
        (p for p in [_pre_base / 'nnunet_preprocessed', _pre_base]
         if (p / f'Dataset{DATASET_ID}_{DATASET_NAME}').exists()),
        _pre_base / 'nnunet_preprocessed'
    )
    # nnUNet eğitimde .npz yanına .npy yazar → /kaggle/input/ salt okunur olduğu
    # için preprocessed'i /kaggle/tmp/'ya kopyalarız (60 GB, yazılabilir).
    NNUNET_PREPROCESSED_P = TMP_DIR / 'nnunet_preprocessed'

    NIFTI_VAL_DIR = next(
        (p for p in [_pre_base / 'nifti_val']
         if p.exists() and any(p.glob('*.nii.gz'))),
        _pre_base / 'nifti_val'
    )
    EGITIM_DIR  = DATA_DIR / 'Egitim Verisi'
    YARISMA_DIR = DATA_DIR / 'Test Verisi'
    BILGI_XLSX  = DATA_DIR / 'Bilgi.xlsx'
    NIFTI_DIR   = TMP_DIR / 'nifti_val_tmp'
    SPLIT_DIR   = WORK_DIR / 'splits'
    RESULTS_P   = TMP_DIR / 'nnunet_results'

elif IS_COLAB:
    WORK_DIR   = Path('/content')
    DRIVE_BASE = Path('/content/drive/MyDrive/Abdomen')

    # abdomen-nnunet Kaggle API ile buraya indirilir (yukarıdaki download hücresi):
    # !kaggle datasets download -d ramazan2020/abdomen-nnunet -p /content/kaggle_data --unzip
    COLAB_PRE_DIR = Path('/content/kaggle_data')

    # Preprocessed: önce /content/kaggle_data/nnunet_preprocessed, sonra Drive
    _pre_candidates = [
        COLAB_PRE_DIR / 'nnunet_preprocessed',
        DRIVE_BASE    / 'nnunet_preprocessed',
        Path('/content/nnunet_preprocessed'),
    ]
    _pre_input = next(
        (p for p in _pre_candidates
         if (p / f'Dataset{DATASET_ID}_{DATASET_NAME}').exists()),
        COLAB_PRE_DIR / 'nnunet_preprocessed'
    )
    NNUNET_PREPROCESSED_P = _pre_input  # Colab'da /content yazılabilir; symlink gereksiz

    # nifti_val: önce indirilenden ara, sonra Drive
    _nifti_candidates = [
        COLAB_PRE_DIR / 'nifti_val',
        DRIVE_BASE    / 'nifti_val',
    ]
    NIFTI_VAL_DIR = next(
        (p for p in _nifti_candidates if p.exists() and any(p.glob('*.nii.gz'))),
        COLAB_PRE_DIR / 'nifti_val'
    )

    # splits: önce indirilenden ara (splits_out veya splits), sonra Drive
    _split_candidates = [
        COLAB_PRE_DIR / 'splits_out',
        COLAB_PRE_DIR / 'splits',
        DRIVE_BASE    / 'splits',
    ]
    SPLIT_DIR = next(
        (p for p in _split_candidates
         if p.exists() and any(p.glob('fold*_train.csv'))),
        DRIVE_BASE / 'splits'
    )

    EGITIM_DIR  = COLAB_PRE_DIR / 'Egitim Verisi'
    YARISMA_DIR = COLAB_PRE_DIR / 'Test Verisi'
    BILGI_XLSX  = COLAB_PRE_DIR / 'Bilgi.xlsx'
    NIFTI_DIR   = WORK_DIR / 'nifti_val_tmp'
    RESULTS_P   = WORK_DIR / 'nnunet_results'

elif IS_LOCAL:
    try:
        from dotenv import load_dotenv; load_dotenv()
    except ImportError: pass
    PROJECT_ROOT          = Path('.').resolve()
    WORK_DIR              = PROJECT_ROOT / 'outputs'
    EGITIM_DIR            = Path(os.environ.get('ABDOMEN_TRAIN_DIR', str(PROJECT_ROOT / 'Egitim Verisi')))
    YARISMA_DIR           = Path(os.environ.get('ABDOMEN_TEST_DIR',  str(PROJECT_ROOT / 'Test Verisi')))
    BILGI_XLSX            = Path(os.environ.get('ABDOMEN_BILGI_XLSX', str(PROJECT_ROOT / 'Bilgi.xlsx')))
    NIFTI_DIR             = WORK_DIR / 'nifti_val_tmp'
    NIFTI_VAL_DIR         = WORK_DIR / 'nifti_val'
    SPLIT_DIR             = WORK_DIR / 'splits'
    _pre_input            = WORK_DIR / 'nnunet_preprocessed'
    NNUNET_PREPROCESSED_P = _pre_input
    RESULTS_P             = WORK_DIR / 'nnunet_results'

RESULTS_P.mkdir(parents=True, exist_ok=True)
NIFTI_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

os.environ['nnUNet_raw']          = str(WORK_DIR / 'nnunet_raw_dummy')
os.environ['nnUNet_preprocessed'] = str(NNUNET_PREPROCESSED_P)
os.environ['nnUNet_results']      = str(RESULTS_P)
os.environ['OMP_NUM_THREADS']     = '1'
os.environ['ABDOMEN_SPLIT_DIR']   = str(SPLIT_DIR)
os.environ['ABDOMEN_TRAIN_DIR']   = str(EGITIM_DIR)
os.environ['ABDOMEN_TEST_DIR']    = str(YARISMA_DIR)
os.environ['ABDOMEN_BILGI_XLSX']  = str(BILGI_XLSX)

print(f'Ortam               : {env_name}')
print(f'CONTINUE_TRAINING   : {CONTINUE_TRAINING}')
print(f'PRE_DATASET_SLUG    : {PRE_DATASET_SLUG}')
if IS_KAGGLE:
    print(f'Preprocessed (input): {_pre_input}  (✓={(_pre_input / f"Dataset{DATASET_ID}_{DATASET_NAME}").exists()})')
print(f'nnUNet_preprocessed : {NNUNET_PREPROCESSED_P}  (✓={(_pre_input / f"Dataset{DATASET_ID}_{DATASET_NAME}").exists()})')
print(f'SPLIT_DIR           : {SPLIT_DIR}  (✓={SPLIT_DIR.exists()})')
print(f'NIFTI_VAL_DIR       : {NIFTI_VAL_DIR}  (✓={NIFTI_VAL_DIR.exists()})')
print(f'RESULTS_P           : {RESULTS_P}')
if IS_KAGGLE:
    print(f'BILGI_XLSX          : {BILGI_XLSX}  (✓={BILGI_XLSX.exists()})')

_plans_input = _pre_input / f'Dataset{DATASET_ID}_{DATASET_NAME}' / 'nnUNetPlans.json'
if not _plans_input.exists():
    raise FileNotFoundError(
        f'nnUNetPlans.json bulunamadı: {_plans_input}\n'
        f'Faz2a\'yı tamamlayıp "{PRE_DATASET_SLUG}" dataset olarak ekleyin.\n'
        f'Colab için: yukarıdaki download hücresini çalıştırın.'
    )
print('nnUNetPlans.json    : ✓')

---
## 3. Preprocessed → Tmp (Symlink)

`/kaggle/input/` salt okunur — nnUNet `.npz` yanına `.npy` yazmaya çalışır ve **OSError** verir.  
Çözüm: `/kaggle/tmp/` altında aynı dizin yapısını oluştur, dosyaları symlink'le. Yalnızca `.npy` tmp'ye yazılır.

In [ ]:
if IS_KAGGLE:
    _src_ds = _pre_input / f'Dataset{DATASET_ID}_{DATASET_NAME}'
    _dst_ds = NNUNET_PREPROCESSED_P / f'Dataset{DATASET_ID}_{DATASET_NAME}'

    _npy_existing = list(_dst_ds.rglob('*.npy')) if _dst_ds.exists() else []
    _npz_total    = list(_src_ds.rglob('*.npz'))

    if _dst_ds.exists() and len(_npy_existing) >= len(_npz_total):
        print(f'Symlink+npy zaten hazır: {len(_npy_existing)} .npy, {len(_npz_total)} .npz')
    else:
        print(f'Symlink yapısı oluşturuluyor: {_src_ds} → {_dst_ds}')
        _linked = _skipped = 0
        for src_f in sorted(_src_ds.rglob('*')):
            if not src_f.is_file():
                continue
            rel   = src_f.relative_to(_src_ds)
            dst_f = _dst_ds / rel
            dst_f.parent.mkdir(parents=True, exist_ok=True)
            if dst_f.exists() or dst_f.is_symlink():
                _skipped += 1
                continue
            os.symlink(src_f, dst_f)
            _linked += 1
        print(f'  ✓  {_linked} symlink oluşturuldu, {_skipped} mevcut')

    # ── nnUNetPlans.json patch_size düzeltmesi ───────────────────────────────
    # Eski format: 'configurations' anahtarı yok; config doğrudan üst seviyede
    #   (örn. _plans['3d_fullres']['patch_size']).
    # Yeni format: _plans['configurations']['3d_fullres']['patch_size'].
    # Stride bilgisi: num_pool_per_axis (eski) → pool_op_kernel_sizes (eski)
    #                 → arch_kwargs.strides (yeni).
    # Hedef: her boyutu 2^n_down ile bölünür yap (decoder skip uyuşmazlığını önler).
    _plans_dst = _dst_ds / 'nnUNetPlans.json'
    if _plans_dst.exists():
        _plans = json.loads(_plans_dst.read_text())
        _patched = False
        if 'configurations' in _plans:
            _plan_cfgs = _plans['configurations']
        else:
            _plan_cfgs = {k: v for k, v in _plans.items()
                          if isinstance(v, dict) and 'patch_size' in v}
        for _cfg_name, _cfg in _plan_cfgs.items():
            _patch = list(_cfg.get('patch_size', []))
            if not _patch:
                continue
            _n_down = list(_cfg.get('num_pool_per_axis', []))
            if not _n_down:
                _ks = (_cfg.get('architecture', {}).get('arch_kwargs', {}).get('strides', [])
                       or _cfg.get('pool_op_kernel_sizes', []))
                if _ks:
                    _n_down = [sum(1 for _s in _ks if _s[_d] > 1)
                               for _d in range(len(_patch))]
            if not _n_down or len(_n_down) != len(_patch):
                print(f'  [{_cfg_name}] stride bilgisi bulunamadı, atlandı')
                continue
            for _d in range(len(_patch)):
                _div = 2 ** _n_down[_d]
                _new = (_patch[_d] // _div) * _div
                if _new != _patch[_d]:
                    print(f'  [{_cfg_name}] patch_size[{_d}]: {_patch[_d]} → {_new}  (2^{_n_down[_d]}={_div})')
                    _patch[_d] = _new
                    _patched   = True
            if _patched:
                _cfg['patch_size'] = _patch
        if _patched:
            if _plans_dst.is_symlink():
                _plans_dst.unlink()
            _plans_dst.write_text(json.dumps(_plans, indent=2))
            print('  ✓  nnUNetPlans.json güncellendi')
        else:
            print('  ✓  nnUNetPlans.json patch_size zaten uyumlu')

    os.environ['nnUNet_preprocessed'] = str(NNUNET_PREPROCESSED_P)
    print(f'nnUNet_preprocessed → {NNUNET_PREPROCESSED_P}')
    _free = shutil.disk_usage(str(TMP_DIR)).free / 1e9
    print(f'Tmp serbest         : {_free:.1f} GB  (.npy dosyaları buraya yazılacak)')
else:
    print('Kaggle dışı — symlink atlandı')

---
## 3. GitHub Repo + Splits

In [ ]:
if IS_LOCAL:
    REPO_DIR = Path('.').resolve()
else:
    REPO_DIR = WORK_DIR / 'abdomen1'
    if not (REPO_DIR / '.git').exists():
        subprocess.run(['git', 'clone', '--depth=1', GITHUB_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], capture_output=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src.splits     import make_splits, raw_case_id
from src.lifting    import BboxLifter
from src.evaluation import f1_at_iou, top5_f1_mean

# ── manifest.csv ─────────────────────────────────────────────────────────────
_manifest_csv = SPLIT_DIR / 'manifest.csv'
if not _manifest_csv.exists():
    print('manifest.csv oluşturuluyor...')
    from src.preprocessing import build_manifest
    build_manifest(_manifest_csv)
    print(f'  ✓  ({_manifest_csv.stat().st_size/1e3:.0f} KB)')
else:
    print(f'manifest.csv mevcut  ({_manifest_csv.stat().st_size/1e3:.0f} KB)')

# ── splits.csv + fold CSV'leri ───────────────────────────────────────────────
_fold_train_csv = SPLIT_DIR / f'fold{FOLD}_train.csv'
_fold_val_csv   = SPLIT_DIR / f'fold{FOLD}_val.csv'
if not _fold_train_csv.exists() or not _fold_val_csv.exists():
    print('Splits oluşturuluyor...')
    make_splits(out_dir=SPLIT_DIR)
    print('  ✓')
else:
    print('splits.csv mevcut')

# load_fold() src/config.py'deki modül-seviye SPLIT_DIR'ı kullandığı için
# env var değişikliğini görmez; doğrudan CSV okuması güvenlidir.
manifest    = pd.read_csv(_manifest_csv)
train_cases = pd.read_csv(_fold_train_csv)['Case Number'].astype(str).tolist()
val_cases   = pd.read_csv(_fold_val_csv)['Case Number'].astype(str).tolist()
all_cases   = sorted(set(train_cases + val_cases))

print(f'Manifest  : {len(manifest):,} satır')
print(f'Fold {FOLD} : {len(train_cases)} train + {len(val_cases)} val = {len(all_cases)} toplam')

def lift_2d_to_3d(manifest_df, case, delta_z=2, iou_th=0.3):
    return BboxLifter(manifest_df, egitim_dir=EGITIM_DIR,
                      yarisma_dir=YARISMA_DIR, delta_z=delta_z, iou_th=iou_th).lift(case)

---
## 4. Checkpoint İndirme / Geri Yükleme

`CONTINUE_TRAINING = True` ise:
1. `CHECKPOINT_URL` doluysa → zip indirilir, `/kaggle/working/nnunet_results/` altına extract edilir
2. URL boşsa → `/kaggle/working/nnunet_results/` içinde mevcut checkpoint aranır

**URL nasıl alınır?**  
Önceki session'da **Save Version** sonrası `Output` sekmesinden `nnunet_results/` klasörünü zip olarak indirin ve direkt link kopyalayın.

In [ ]:
import zipfile, urllib.request, shutil

# ─── Checkpoint kaynağı ──────────────────────────────────────────────────────
# Kaggle'da önceki session checkpoint'ini devam ettirmek için iki yol:
#
# YOL A — Dataset olarak ekle (önerilen, hızlı):
#   1. Önceki session → Save Version → çıktılar dataset olarak kaydedilir
#   2. Bu notebook'a sağ panelden "Add Dataset" → nnunet-checkpoint slug'ı
#   3. CKPT_DATASET_SLUG'ı gerçek slug ile eşleştir, CHECKPOINT_URL='', CONTINUE_TRAINING=True
#
# YOL B — Direkt URL (Kaggle dışından zip link):
#   CHECKPOINT_URL = 'https://...'  (Save Version → Output → Download → URL kopyala)

CKPT_DATASET_SLUG = 'nnunet-checkpoint'  # Kaggle'a eklenen dataset slug'ı

_ckpt_dst = RESULTS_P / f'Dataset{DATASET_ID}_{DATASET_NAME}'

if not CONTINUE_TRAINING:
    print('Yeni eğitim (CONTINUE_TRAINING=False) — checkpoint atlandı')

else:
    _restored = False

    # ── Yol A: Kaggle input dataset ─────────────────────────────────────────
    if IS_KAGGLE and not CHECKPOINT_URL:
        _ckpt_base = Path('/kaggle/input/datasets/ramazan2020')
        # Slug önce tam yolda, sonra alternatif konumlarda ara
        _ckpt_candidates = [
            _ckpt_base / CKPT_DATASET_SLUG,
            Path('/kaggle/input') / CKPT_DATASET_SLUG,
        ]
        _ckpt_input = next((p for p in _ckpt_candidates if p.exists()), None)

        if _ckpt_input:
            print(f'Dataset checkpoint bulundu: {_ckpt_input}')
            # nnunet_results/Dataset100_Abdomen/ veya Dataset100_Abdomen/ yapısı olabilir
            _src_candidates = [
                _ckpt_input / 'nnunet_results' / f'Dataset{DATASET_ID}_{DATASET_NAME}',
                _ckpt_input / f'Dataset{DATASET_ID}_{DATASET_NAME}',
            ]
            _ckpt_src = next((p for p in _src_candidates if p.exists()), None)

            if _ckpt_src:
                print(f'  Kaynak: {_ckpt_src}')
                # Doğrudan symlink — /kaggle/input/ salt okunur ama checkpoint sadece okunur
                # nnUNet devamda yeni checkpoint yazar → RESULTS_P içinde olmalı
                if _ckpt_dst.exists() or _ckpt_dst.is_symlink():
                    shutil.rmtree(str(_ckpt_dst), ignore_errors=True)
                # Kopyala (input salt okunur, checkpoint yazma işlemi için /working gerekli)
                print('  Kopyalanıyor (yazılabilir hedef için gerekli)...')
                shutil.copytree(str(_ckpt_src), str(_ckpt_dst))
                _restored = True
                print(f'  ✓  {_ckpt_dst}  ({sum(1 for _ in _ckpt_dst.rglob("*.pth"))} .pth dosyası)')
            else:
                print(f'  ⚠ Dataset bulundu ama Dataset{DATASET_ID}_{DATASET_NAME}/ klasörü yok')
                print(f'  Mevcut içerik: {list(_ckpt_input.iterdir())[:5]}')
        else:
            print(f'Kaggle dataset bulunamadı (slug: {CKPT_DATASET_SLUG})')
            print('  → "Add Data" panelinden nnunet-checkpoint dataset ekleyin')
            print(f'  → veya CHECKPOINT_URL doldurun')

    # ── Yol B: URL'den indir ────────────────────────────────────────────────
    if not _restored and CHECKPOINT_URL:
        _zip_path = (TMP_DIR if IS_KAGGLE else WORK_DIR) / 'nnunet_checkpoint.zip'
        print(f'URL\'den indiriliyor: {CHECKPOINT_URL}')
        urllib.request.urlretrieve(CHECKPOINT_URL, str(_zip_path))
        print(f'  İndirildi ({_zip_path.stat().st_size / 1e6:.0f} MB), extract ediliyor...')
        with zipfile.ZipFile(str(_zip_path), 'r') as zf:
            _names = zf.namelist()
            if any(n.startswith('nnunet_results/') for n in _names):
                zf.extractall(str(WORK_DIR))
            else:
                zf.extractall(str(RESULTS_P))
        _zip_path.unlink(missing_ok=True)
        _restored = _ckpt_dst.exists()
        print(f'  ✓  {_ckpt_dst}  (exists={_restored})')

    # ── Yol C: /working içinde checkpoint var mı ────────────────────────────
    if not _restored:
        _trainer_sub = 'nnUNetTrainer__nnUNetPlans__3d_fullres'
        _fold_dir    = _ckpt_dst / _trainer_sub / f'fold_{FOLD}'
        _ptfiles     = list(_fold_dir.glob('checkpoint_*.pth')) if _fold_dir.exists() else []
        if _ptfiles:
            print(f'Mevcut checkpoint /working içinde bulundu: {_fold_dir}  ({len(_ptfiles)} .pth)')
            _restored = True
        else:
            print(f'⚠ Checkpoint bulunamadı: {_fold_dir}')
            print('  Seçenekler:')
            print(f'  A) CKPT_DATASET_SLUG="{CKPT_DATASET_SLUG}" dataset ekleyin (Add Data)')
            print('  B) CHECKPOINT_URL doldurun')
            print('  C) CONTINUE_TRAINING=False yapın (sıfırdan başlatır)')

---
## 5. Eğitim

**Kaggle 9 saat session sınırı:** Checkpoint otomatik kaydedilir.  
Session bitince: çıktıyı `nnunet-checkpoint` dataset olarak yayınlayın, yeni session'da `CONTINUE_TRAINING = True` yapın.

In [ ]:
import torch, pathlib, importlib.util, py_compile as _pyc_mod, re as _re
assert torch.cuda.is_available(), 'GPU gerekli!'

NUM_EPOCHS = 300   # varsayılan nnUNet: 1000

os.environ['nnUNet_compile'] = '0'

_dl_spec = importlib.util.find_spec('nnunetv2.training.dataloading.data_loader')
if _dl_spec and _dl_spec.origin:
    _dl_path = pathlib.Path(_dl_spec.origin)
    _dl_src  = _dl_path.read_text()
    _dl_new  = _dl_src

    _dl_new = _dl_new.replace("mmap_mode=None", "mmap_mode='r'")
    _dl_new = _dl_new.replace("mmap_mode='c'",  "mmap_mode='r'")
    _dl_new = _re.sub(
        r'torch\.from_numpy\((crop_and_pad_nd\([^)]+\))\)',
        r'torch.from_numpy(\1.copy())',
        _dl_new
    )

    if _dl_new != _dl_src:
        _dl_path.write_text(_dl_new)
        _pycache = _dl_path.parent / '__pycache__'
        for _pyc in _pycache.glob('data_loader*.pyc'):
            _pyc.unlink(missing_ok=True)
        _pyc_mod.compile(str(_dl_path), doraise=True)
        print("data_loader.py patched ✓")
        for _line in _dl_new.split('\n'):
            if 'mmap_mode' in _line or '.copy())' in _line:
                print(f"  {_line.strip()}")
    else:
        print("data_loader.py zaten düzeltilmiş")

_args = [find_nnunet_cmd('nnUNetv2_train'),
         str(DATASET_ID), '3d_fullres', str(FOLD),
         '-num_epochs', str(NUM_EPOCHS),
         '--npz']
if CONTINUE_TRAINING:
    _args.append('--c')

print(f'=== nnUNet Eğitim ===')
print(f'GPU     : {torch.cuda.get_device_name(0)}')
print(f'Dataset : {DATASET_ID}  Config: 3d_fullres  Fold: {FOLD}  Devam: {CONTINUE_TRAINING}')
print(f'Epoch   : {NUM_EPOCHS}  (~{NUM_EPOCHS * 78 / 3600:.1f} saat)')
print(f'Results : {RESULTS_P}')
print('─' * 50)

_proc = subprocess.Popen(
    _args, env={**os.environ},
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    universal_newlines=True
)
for line in _proc.stdout:
    print(line, end='', flush=True)
_proc.wait()

print('─' * 50)
if _proc.returncode == 0:
    print('Eğitim tamamlandı ✓')
else:
    print(f'Eğitim kodu: {_proc.returncode} (session kesildi / hata)')
    print('Checkpoint nnunet_results/ altında — Save Version ile kaydedin')

if IS_COLAB:
    _src = RESULTS_P / f'Dataset{DATASET_ID}_{DATASET_NAME}'
    _dst = Path('/content/drive/MyDrive/Abdomen/nnunet_results') / f'Dataset{DATASET_ID}_{DATASET_NAME}'
    if _src.exists():
        _dst.parent.mkdir(parents=True, exist_ok=True)
        if _dst.exists(): shutil.rmtree(str(_dst))
        shutil.copytree(str(_src), str(_dst))
        print(f"Drive'a yedeklendi: {_dst}")

---
## 6. Val NIfTI Hazırlama

In [ ]:
import SimpleITK as sitk
import pydicom
from concurrent.futures import ThreadPoolExecutor
from tqdm.notebook import tqdm

VAL_INPUT_DIR = WORK_DIR / 'val_input'
VAL_INPUT_DIR.mkdir(parents=True, exist_ok=True)

_linked = 0
_missing_cases = []

for c in val_cases:
    raw = raw_case_id(c)
    dst = VAL_INPUT_DIR / f'case_{raw:05d}_0000.nii.gz'
    if dst.exists(): continue
    src = NIFTI_VAL_DIR / f'case_{raw:05d}_0000.nii.gz'
    if src.exists():
        try: os.symlink(src, dst)
        except (OSError, NotImplementedError): shutil.copy2(str(src), str(dst))
        _linked += 1
    else:
        _missing_cases.append(c)

print(f'Faz2a nifti_val: {_linked} linklendi  |  eksik: {len(_missing_cases)}')

if _missing_cases and EGITIM_DIR.exists():
    print(f'{len(_missing_cases)} vaka DICOM\'dan üretiliyor...')

    def _dicom_to_nifti(case: str) -> str:
        raw = raw_case_id(case)
        out = NIFTI_DIR / f'case_{raw:05d}_0000.nii.gz'
        if out.exists(): return 'skip'
        case_dir = EGITIM_DIR / str(raw)
        if not case_dir.exists(): return f'missing:{case}'
        reader = sitk.ImageSeriesReader()
        names  = reader.GetGDCMSeriesFileNames(str(case_dir))
        if not names: return f'no_dcm:{case}'
        size_map = {}
        for n in names:
            try:
                h = pydicom.dcmread(n, stop_before_pixels=True)
                size_map.setdefault((int(h.Rows), int(h.Columns)), []).append(n)
            except Exception: pass
        if len(size_map) > 1:
            names = max(size_map.values(), key=len)
        reader.SetFileNames(names)
        try:
            sitk.WriteImage(reader.Execute(), str(out))
            return 'ok'
        except Exception as e:
            return f'err:{e}'

    sitk.ProcessObject.GlobalWarningDisplayOff()
    with ThreadPoolExecutor(max_workers=4) as ex:
        _res = list(tqdm(ex.map(_dicom_to_nifti, _missing_cases),
                         total=len(_missing_cases), desc='Val DICOM→NIfTI'))
    sitk.ProcessObject.GlobalWarningDisplayOn()

    for c in _missing_cases:
        raw = raw_case_id(c)
        src = NIFTI_DIR / f'case_{raw:05d}_0000.nii.gz'
        dst = VAL_INPUT_DIR / f'case_{raw:05d}_0000.nii.gz'
        if src.exists() and not dst.exists():
            try: os.symlink(src, dst)
            except (OSError, NotImplementedError): shutil.copy2(str(src), str(dst))
elif _missing_cases:
    print(f'⚠ {len(_missing_cases)} NIfTI eksik ve ham DICOM dataset yok')
    print(f'  Sağ panelden "{KAGGLE_DATASET_SLUG}" ekleyin')

_ready = len(list(VAL_INPUT_DIR.glob('*.nii.gz')))
print(f'Val input hazır: {_ready}/{len(val_cases)} dosya')

---
## 7. Inference — Validasyon Seti

In [ ]:
# Hangi checkpoint kullanılacak (.pth dahil tam ad):
#   checkpoint_latest.pth → eğitim devam ederken ara değerlendirme
#   checkpoint_best.pth   → EMA Dice en yüksek nokta
#   checkpoint_final.pth  → 1000. epoch sonrası (eğitim bitince)
CKPT_NAME = 'checkpoint_latest.pth'

_fold_dir = (RESULTS_P / f'Dataset{DATASET_ID}_{DATASET_NAME}'
             / 'nnUNetTrainer__nnUNetPlans__3d_fullres' / f'fold_{FOLD}')

# Mevcut checkpoint'leri göster
print('Mevcut checkpoint\'ler:')
for _ckpt in sorted(_fold_dir.glob('checkpoint_*.pth')):
    _mb = _ckpt.stat().st_size / 1e6
    _marker = ' ← seçili' if _ckpt.name == CKPT_NAME else ''
    print(f'  {_ckpt.name}  ({_mb:.0f} MB){_marker}')

_ckpt_path = _fold_dir / CKPT_NAME
if not _ckpt_path.exists():
    raise FileNotFoundError(
        f'{CKPT_NAME} bulunamadı: {_ckpt_path}\n'
        f'CKPT_NAME değişkenini mevcut bir checkpoint ile güncelleyin.'
    )

VAL_OUTPUT_DIR = _fold_dir / 'val_predictions'
VAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'\nVal input  : {len(list(VAL_INPUT_DIR.glob("*.nii.gz")))} dosya')
print(f'Checkpoint : {CKPT_NAME}')
print(f'Çıktı      : {VAL_OUTPUT_DIR}')
print('Inference başlatılıyor...')

_proc = subprocess.Popen(
    [find_nnunet_cmd('nnUNetv2_predict'),
     '-i', str(VAL_INPUT_DIR), '-o', str(VAL_OUTPUT_DIR),
     '-d', str(DATASET_ID), '-c', '3d_fullres', '-f', str(FOLD),
     '-chk', CKPT_NAME,
     '--save_probabilities'],
    env={**os.environ},
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    universal_newlines=True
)
for line in _proc.stdout:
    print(line, end='', flush=True)
_proc.wait()
print(f'Inference tamamlandı: {len(list(VAL_OUTPUT_DIR.glob("*.nii.gz")))} mask')

---
## 8. Değerlendirme — F1 Skoru

In [ ]:
from scipy import ndimage

def seg_to_pred_df(pred_dir: Path) -> pd.DataFrame:
    rows = []
    for nii_path in sorted(pred_dir.glob('*.nii.gz')):
        try: raw = int(nii_path.stem.split('_')[1])
        except: continue
        mask      = sitk.GetArrayFromImage(sitk.ReadImage(str(nii_path)))
        prob_path = nii_path.with_suffix('').with_suffix('.npz')
        probs     = np.load(str(prob_path))['probabilities'] if prob_path.exists() else None
        for label_id in range(1, len(SUPER_CLASSES)+1):
            binary = (mask == label_id).astype(np.uint8)
            if binary.sum() == 0: continue
            labeled, n_comp = ndimage.label(binary)
            total_vox = float(binary.sum())
            for comp_id in range(1, n_comp+1):
                cm = (labeled == comp_id)
                coords = np.where(cm)
                z1,z2 = int(coords[0].min()), int(coords[0].max())
                y1,y2 = int(coords[1].min()), int(coords[1].max())
                x1,x2 = int(coords[2].min()), int(coords[2].max())
                score = float(probs[label_id][cm].mean()) if probs is not None else float(cm.sum())/total_vox
                for z in range(z1, z2+1):
                    rows.append({'case':raw,'image_id':z,'class':label_id-1,'score':score,
                                 'x1':float(x1),'y1':float(y1),'x2':float(x2),'y2':float(y2)})
    return pd.DataFrame(rows)

pred_df = seg_to_pred_df(VAL_OUTPUT_DIR)
print(f'Prediction: {len(pred_df):,} satır, {pred_df["case"].nunique() if not pred_df.empty else 0} vaka')

gt_rows = []
for case in val_cases:
    raw = raw_case_id(case)
    for b in lift_2d_to_3d(manifest, case):
        for z in range(int(b['z1']), int(b['z2'])+1):
            gt_rows.append({'case':raw,'image_id':z,'class':int(b['class']),
                            'x1':float(b['x1']),'y1':float(b['y1']),
                            'x2':float(b['x2']),'y2':float(b['y2'])})
gt_df = pd.DataFrame(gt_rows)

if pred_df.empty:
    print('Prediction boş — inference adımını kontrol edin')
else:
    top5   = top5_f1_mean(pred_df, gt_df)
    detail = f1_at_iou(pred_df, gt_df, iou_th=0.3)

    print(f'\n=== nnUNet v2 — Fold {FOLD} ===')
    print(f'Top-5 Mean F1 : {top5["top5_mean_f1"]:.4f}')
    print(f'Macro F1 @0.3 : {detail["macro_f1"]:.4f}')
    print(f'Micro F1 @0.3 : {detail["micro_f1"]:.4f}')
    print()
    for cls_id, cls_name in enumerate(SUPER_CLASSES):
        if cls_id in detail['per_class']:
            m = detail['per_class'][cls_id]
            print(f'  {cls_id} {cls_name:<30} P={m["precision"]:.3f} R={m["recall"]:.3f} F1={m["f1"]:.3f}')

---
## 9. Sonuçları Kaydet

In [ ]:
OUTPUT_DIR = WORK_DIR / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not pred_df.empty:
    pred_df.to_csv(OUTPUT_DIR / f'pred_fold{FOLD}.csv', index=False)
    gt_df.to_csv(OUTPUT_DIR   / f'gt_fold{FOLD}.csv',   index=False)
    summary = {
        'model': 'nnUNet_3d_fullres', 'fold': FOLD, 'val_cases': len(val_cases),
        'top5_mean_f1': top5['top5_mean_f1'],
        'macro_f1_03' : detail['macro_f1'],
        'micro_f1_03' : detail['micro_f1'],
        'per_threshold': {str(th): float(f1v) for th, f1v in top5['per_threshold']},
    }
    (OUTPUT_DIR / f'summary_fold{FOLD}.json').write_text(json.dumps(summary, indent=2, default=float))

print(f'Çıktılar: {OUTPUT_DIR}')
for f in sorted(OUTPUT_DIR.glob('*')):
    print(f'  {f.name}  ({f.stat().st_size/1e3:.0f} KB)')

if IS_COLAB:
    _dst = Path('/content/drive/MyDrive/Abdomen/output_nnunet')
    if _dst.exists(): shutil.rmtree(str(_dst))
    shutil.copytree(str(OUTPUT_DIR), str(_dst))
    print(f"Drive'a kopyalandı: {_dst}")

print('\nTamamlandı!')
if IS_KAGGLE:
    print('Save Version → checkpoint + sonuçlar /kaggle/working/ altında saklanır')

---
## 10. Çıktıları Zip'le & İndir

In [ ]:
import zipfile, time

_zip_targets = {
    'nnunet_results': RESULTS_P,
    'output':         WORK_DIR / 'output',
    'splits':         SPLIT_DIR,
}

_ts       = time.strftime('%Y%m%d_%H%M')
_zip_name = f'nnunet_fold{FOLD}_{_ts}.zip'
_zip_path = WORK_DIR / _zip_name

print(f'Zip oluşturuluyor: {_zip_path}')
with zipfile.ZipFile(str(_zip_path), 'w', zipfile.ZIP_DEFLATED, compresslevel=1) as zf:
    for _label, _folder in _zip_targets.items():
        if not _folder.exists():
            print(f'  [{_label}] yok, atlandı')
            continue
        _count = 0
        for _f in sorted(_folder.rglob('*')):
            if not _f.is_file():
                continue
            zf.write(str(_f), str(Path(_label) / _f.relative_to(_folder)))
            _count += 1
        print(f'  [{_label}] {_count} dosya')

_zip_size_mb = _zip_path.stat().st_size / 1e6
print(f'\nZip hazır: {_zip_name}  ({_zip_size_mb:.0f} MB)')
print()
print('⚠ Önce Save Version yapın, sonra aşağıdaki hücreyi çalıştırın.')
print('  1. Sağ üst → Save Version')
print('  2. Output sekmesi → zip → ⋮ → Download')
print('  3. Zip sil hücresini çalıştır → disk alanı geri kazanılır')

In [ ]:
# Save Version + indirme bittikten sonra çalıştırın
for _zf in sorted((WORK_DIR).glob('nnunet_fold*.zip')):
    _mb = _zf.stat().st_size / 1e6
    _zf.unlink()
    print(f'Silindi: {_zf.name}  ({_mb:.0f} MB)')

print(f'Working serbest: {shutil.disk_usage(str(WORK_DIR)).free / 1e9:.1f} GB')

---
## 11. Tmp'den Working'e Kopyala & İndir

In [ ]:
import shutil

# /kaggle/tmp/ → /kaggle/working/ (Output sekmesinden indirilebilir hale getirir)
_src = Path('/kaggle/tmp/nnunet_fold0_20260616_0816.zip')
_dst = Path('/kaggle/working') / _src.name

if not _src.exists():
    print(f'Dosya bulunamadı: {_src}')
    # /kaggle/tmp/ içindeki zip'leri listele
    _zips = list(Path('/kaggle/tmp').glob('*.zip'))
    if _zips:
        print('Mevcut zip dosyaları:')
        for _z in sorted(_zips):
            print(f'  {_z}  ({_z.stat().st_size / 1e6:.0f} MB)')
else:
    if _dst.exists():
        print(f'Zaten mevcut: {_dst.name}')
    else:
        print(f'Kopyalanıyor ({_src.stat().st_size / 1e6:.0f} MB)...')
        shutil.copy2(str(_src), str(_dst))
        print(f'✓ {_dst}')

    print()
    print('İndirme adımları:')
    print('  1. Sağ üst → Save Version')
    print(f'  2. Output sekmesi → {_dst.name} → ⋮ → Download')

In [ ]:
# Zip working'e kopyalandıktan ve Save Version yapıldıktan sonra çalıştırın
_to_delete = {
    'nnunet_results': RESULTS_P,
    'output':         WORK_DIR / 'output',
    'splits':         SPLIT_DIR,
}
for _label, _folder in _to_delete.items():
    if _folder.exists():
        shutil.rmtree(str(_folder))
        print(f'Silindi [{_label}]: {_folder}')
    else:
        print(f'Yok [{_label}]: {_folder}')

print(f'Working serbest: {shutil.disk_usage(str(WORK_DIR)).free / 1e9:.1f} GB')